### Notebook 04: Robustness & External Validation

**CS 210 Final Project** | What the Headlines Say: Measuring Cognitive Bias and Agency Framing in AI News Discourse | Sadhana Vasanthakumar

Two forms of external validation that notebooks 01–03 can't provide from within the AI corpus alone.

**PSI hand-validation** compares the automated PSI classifier against human judgments on the 200-headline sample labeled during notebook 02. Produces Cohen's kappa and per-class accuracy. Even fair agreement (kappa > 0.40) is a defensible claim that PSI measures what it says.

**Invisible Human control corpus comparison** scrapes ~4,000 climate change headlines and runs the identical IH vocabulary pipeline. If AI headlines score lower than climate on most domains, the 92% absence finding is specific to AI coverage. If they're similar, headline compression explains it and the framing needs adjusting. Both outcomes are honest findings — the code labels the result accordingly.

#### Inputs

| File | Source | Required |
|------|--------|----------|
| `data/bias_observatory.db` | Notebook 01 output | Yes |
| `outputs/features_for_modeling.csv` | Notebook 02 output | Yes |
| `outputs/gold_standard_labeled.csv` | Hand-labeled in notebook 02 cell 10 | Recommended |

#### Outputs

| File | Purpose |
|------|---------|
| `outputs/psi_validation_results.csv` | Kappa + per-class accuracy |
| `outputs/control_corpus_headlines.csv` | Climate change control headlines |
| `outputs/ih_comparison.csv` | AI vs control IH rates per domain |
| `outputs/figures/12_ih_comparison.png` | Side-by-side IH bar chart |
| `outputs/validation_summary.json` | All validation numbers |

#### Dependencies

```bash
pip install nltk pandas numpy matplotlib seaborn gnews scipy scikit-learn
```

#### How to run

Takes 20–35 minutes end-to-end — mostly the control corpus scrape in cell 3 (10–20 minutes depending on gnews rate limits). If `outputs/control_corpus_headlines.csv` already exists from a previous run, cell 3 skips the scrape automatically.

1. Confirm `data/bias_observatory.db` and `outputs/features_for_modeling.csv` exist from notebooks 01 and 02
2. Run cell 1 (imports and load)
3. Run cell 2 — computes kappa from `gold_standard_labeled.csv` (already labeled during notebook 02)
4. Run cell 3 — control corpus scrape. Safe to interrupt, it checkpoints after every query and resumes where it left off
5. Run cell 4 — IH comparison (2–3 minutes)
6. Run cell 5 — exports validation summary

In [1]:
# imports and load corpus features. reuses existing outputs — nothing
# expensive runs in this notebook.

import sqlite3
import pandas as pd
import numpy as np
import os
import re
import json
import time
import warnings
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import cohen_kappa_score, classification_report
from scipy.stats import spearmanr, chi2_contingency

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs('outputs/figures', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

DB_PATH = 'data/bias_observatory.db'

# load AI corpus features (produced by notebook 02)
df = pd.read_csv('outputs/features_for_modeling.csv', encoding='utf-8')

# load raw headlines for PSI validation (need title + psi_flag)
conn = sqlite3.connect(DB_PATH)
raw = pd.read_sql_query(
    'SELECT r.id, r.title, r.publisher, r.window, f.psi_flag, f.psi_score '
    'FROM headlines_raw r '
    'JOIN headlines_features f ON r.id = f.headline_id',
    conn
)
conn.close()

assert len(raw) > 100_000, f'Expected >100K rows, got {len(raw):,}'

print(f'loaded {len(df):,} feature rows')
print(f'loaded {len(raw):,} raw headlines with PSI flags')
print()
print('PSI flag distribution in full corpus:')
print(raw['psi_flag'].value_counts().to_string())
print()
print(f'run started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

loaded 179,372 feature rows
loaded 179,372 raw headlines with PSI flags

PSI flag distribution in full corpus:
psi_flag
NEUTRAL        169387
AI_AGENT         8620
HUMAN_AGENT      1176
DUAL_AGENT        189

run started: 2026-05-04 19:32:25


In [2]:
# PSI hand-validation: compute Cohen's kappa from gold_standard_labeled.csv.
# this file was labeled during notebook 02 cell 10 — no additional work needed.
# kappa goes in the report's methodology section to show PSI measures what it claims.
#
# kappa interpretation: <0.20 poor, 0.21-0.40 fair, 0.41-0.60 moderate,
# 0.61-0.80 substantial, >0.80 excellent

GOLD_LABELED_PATH = 'outputs/gold_standard_labeled.csv'
RESULTS_PATH      = 'outputs/psi_validation_results.csv'
VALID_LABELS = {'AI_AGENT', 'HUMAN_AGENT', 'NEUTRAL'}

if not os.path.exists(GOLD_LABELED_PATH):
    print(f'file not found: {GOLD_LABELED_PATH}')
    print('run notebook 02 cell 10 to generate and label the gold standard file.')
    psi_validation_result = {'status': 'not_run', 'kappa': None}
else:
    labeled = pd.read_csv(GOLD_LABELED_PATH, encoding='utf-8')
    labeled['psi_label']  = labeled['psi_label'].astype(str).str.strip().str.upper()
    labeled['model_psi']  = labeled['model_psi'].astype(str).str.strip().str.upper()

    evaluated = labeled[labeled['psi_label'].isin(VALID_LABELS)].copy()

    print(f'total rows in file: {len(labeled)}')
    print(f'rows with psi_label filled: {len(evaluated)}')
    print()

    if len(evaluated) < 20:
        print('too few labeled rows (need >= 20). label more rows in gold_standard_labeled.csv.')
        psi_validation_result = {'status': 'insufficient_labels', 'n_labeled': len(evaluated)}
    else:
        kappa    = cohen_kappa_score(evaluated['psi_label'], evaluated['model_psi'])
        accuracy = (evaluated['psi_label'] == evaluated['model_psi']).mean()

        if kappa < 0.20:   kappa_interp = 'poor'
        elif kappa < 0.40: kappa_interp = 'fair'
        elif kappa < 0.60: kappa_interp = 'moderate'
        elif kappa < 0.80: kappa_interp = 'substantial'
        else:              kappa_interp = 'excellent'

        print('PSI hand-validation results:')
        print(f'   n labeled:        {len(evaluated)}')
        print(f'   Cohen kappa:      {kappa:.4f}  ({kappa_interp} agreement)')
        print(f'   overall accuracy: {accuracy:.4f}')
        print()
        print('confusion matrix (rows=your label, cols=model label):')
        print(pd.crosstab(
            evaluated['psi_label'],
            evaluated['model_psi'],
            rownames=['YOUR LABEL'],
            colnames=['MODEL'],
        ).to_string())
        print()
        print('per-class breakdown:')
        print(classification_report(
            evaluated['psi_label'],
            evaluated['model_psi'],
            labels=sorted(VALID_LABELS),
            zero_division=0,
        ))
        print()
        print(f'PSI was validated against human judgments on a stratified')
        print(f'{len(evaluated)}-headline sample. Cohen kappa = {kappa:.2f},')
        print(f'indicating {kappa_interp} agreement.')
        if kappa < 0.40:
            print('given this level of agreement, PSI findings are presented')
            print('as exploratory rather than definitive.')
        elif kappa < 0.60:
            print('this meets the fair-to-moderate threshold for lexicon-based NLP.')
        else:
            print('this meets the substantial-agreement threshold, supporting')
            print('PSI as a reliable agency framing indicator.')

        results_df = evaluated[['id', 'title', 'psi_label', 'model_psi']].copy()
        results_df['match'] = results_df['psi_label'] == results_df['model_psi']
        results_df.to_csv(RESULTS_PATH, index=False, encoding='utf-8')
        print()
        print(f'saved: {RESULTS_PATH}')

        psi_validation_result = {
            'status':       'complete',
            'n_labeled':    int(len(evaluated)),
            'kappa':        round(float(kappa), 4),
            'kappa_interp': kappa_interp,
            'accuracy':     round(float(accuracy), 4),
        }

print()
print('stored in psi_validation_result for export in cell 5.')

total rows in file: 200
rows with psi_label filled: 200

PSI hand-validation results:
   n labeled:        200
   Cohen kappa:      0.3680  (fair agreement)
   overall accuracy: 0.6000

confusion matrix (rows=your label, cols=model label):
MODEL        AI_AGENT  HUMAN_AGENT  NEUTRAL
YOUR LABEL                                 
AI_AGENT           21            0       24
HUMAN_AGENT         4           24       49
NEUTRAL             2            1       75

per-class breakdown:
              precision    recall  f1-score   support

    AI_AGENT       0.78      0.47      0.58        45
 HUMAN_AGENT       0.96      0.31      0.47        77
     NEUTRAL       0.51      0.96      0.66        78

    accuracy                           0.60       200
   macro avg       0.75      0.58      0.57       200
weighted avg       0.74      0.60      0.57       200


PSI was validated against human judgments on a stratified
200-headline sample. Cohen kappa = 0.37,
indicating fair agreement.
given this

In [3]:
# control corpus scrape: climate change headlines via gnews.
# climate change chosen as control because it shares key properties with
# AI news: long-running story, mix of science/policy/economy/human impact,
# similar average headline length and publisher diversity.
# checkpoint saves after every query so it's safe to interrupt and resume.
# if the output file already exists, the entire scrape is skipped.
# estimated runtime: 10-20 minutes

CONTROL_PATH    = "outputs/control_corpus_headlines.csv"
CHECKPOINT_PATH = "outputs/control_corpus_checkpoint.csv"

CONTROL_QUERIES = [
    "climate change",
    "climate crisis",
    "global warming",
    "carbon emissions",
    "renewable energy climate",
    "climate policy",
    "climate science",
    "net zero",
    "fossil fuels climate",
    "climate adaptation",
    "extreme weather climate",
    "sea level rise",
    "climate health impact",
    "climate jobs workers",
    "climate technology solutions",
]

if os.path.exists(CONTROL_PATH):
    control_df = pd.read_csv(CONTROL_PATH, encoding="utf-8")
    print(f"control corpus already exists: {len(control_df):,} headlines")
    print(f"loaded from {CONTROL_PATH} -- skipping scrape.")
else:
    try:
        from gnews import GNews

        gn = GNews(language="en", country="US", max_results=500)
        all_articles = []

        if os.path.exists(CHECKPOINT_PATH):
            ckpt = pd.read_csv(CHECKPOINT_PATH)
            all_articles = ckpt.to_dict("records")
            completed_queries = set(ckpt["query"].unique())
            print(f"resuming from checkpoint: {len(all_articles):,} articles, "
                  f"{len(completed_queries)} queries done")
        else:
            completed_queries = set()

        for query in CONTROL_QUERIES:
            if query in completed_queries:
                print(f"   skipping (already done): {query}")
                continue
            print(f"   scraping: '{query}'...")
            try:
                encoded_query = query.replace(" ", "+")
                articles = gn.get_news(encoded_query)
                for a in articles:
                    all_articles.append({
                        "title":     a.get("title", ""),
                        "publisher": a.get("publisher", {}).get("title", ""),
                        "date":      str(a.get("published date", "")),
                        "query":     query,
                    })
                print(f"      got {len(articles)} articles (total: {len(all_articles):,})")
                pd.DataFrame(all_articles).to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8")
                time.sleep(2)
            except Exception as e:
                print(f"      error on '{query}': {e} -- skipping")
                time.sleep(5)

        control_df = pd.DataFrame(all_articles)

        if control_df.empty or "title" not in control_df.columns:
            print("no articles scraped. gnews may be rate-limited or not installed.")
            print("run: pip install gnews  then re-run this cell.")
            control_df = pd.DataFrame(columns=["title", "publisher", "date", "query", "title_lower"])
        else:
            control_df = control_df[control_df["title"].str.strip() != ""].copy()
            control_df = control_df.drop_duplicates(subset="title").reset_index(drop=True)
            control_df["title_lower"] = control_df["title"].str.lower().str.strip()
            control_df.to_csv(CONTROL_PATH, index=False, encoding="utf-8")
            print(f"\ncontrol corpus saved: {len(control_df):,} unique headlines")

    except ImportError:
        print("gnews not installed. run: pip install gnews")
        print("or place a pre-scraped CSV at outputs/control_corpus_headlines.csv")
        print("required columns: title, publisher, date, query")
        control_df = pd.DataFrame(columns=["title", "publisher", "date", "query", "title_lower"])

if len(control_df) > 0 and "title" in control_df.columns:
    if "title_lower" not in control_df.columns:
        control_df["title_lower"] = control_df["title"].str.lower().str.strip()
    print()
    print(f"control corpus: {len(control_df):,} headlines")
    print(f"unique publishers: {control_df['publisher'].nunique():,}")
    print(f"queries covered:   {control_df['query'].nunique()}")

control corpus already exists: 4,106 headlines
loaded from outputs/control_corpus_headlines.csv -- skipping scrape.

control corpus: 4,106 headlines
unique publishers: 1,470
queries covered:   15


In [4]:
# Invisible Human: AI corpus vs climate change control.
# runs the IDENTICAL IH vocabulary from notebook 02 cell 8 on the control.
# computes per-domain coverage rates and chi-square significance tests.
#
# two possible outcomes:
#   AI lower on >= 5 of 7 domains: absence is specific to AI coverage
#   AI similar to climate: headline compression explains the absence
# both are honest, reportable findings — the code labels the result accordingly.

# IH vocabulary — identical to notebook 02 cell 8
IH_VOCAB = {
    'ih_grief_loss':           ['grief', 'mourning', 'loss', 'bereavement', 'death',
                                 'dying', 'funeral', 'widow', 'orphan'],
    'ih_spirituality_meaning': ['spiritual', 'faith', 'religion', 'meaning',
                                 'purpose', 'soul', 'transcendence', 'sacred'],
    'ih_love_relationships':   ['love', 'romance', 'friendship', 'family',
                                 'loneliness', 'connection', 'intimacy', 'belonging'],
    'ih_bodily_experience':    ['pain', 'pleasure', 'embodied', 'physical',
                                 'touch', 'sensation', 'illness', 'disability'],
    'ih_dignity_purpose':      ['dignity', 'meaning of work', 'identity',
                                 'self-worth', 'fulfillment', 'calling'],
    'ih_community_belonging':  ['community', 'neighborhood', 'solidarity',
                                 'togetherness', 'shared', 'collective', 'belonging'],
    'ih_childhood_play':       ['childhood', 'children', 'play', 'imagination',
                                 'wonder', 'innocence', 'learning', 'growth'],
}
IH_COLS = list(IH_VOCAB.keys())


def score_ih(titles_lower):
    results = {}
    for col, vocab in IH_VOCAB.items():
        results[col] = titles_lower.apply(
            lambda t: int(any(w in str(t) for w in vocab))
        )
    return pd.DataFrame(results)


if len(control_df) < 100:
    print('control corpus too small or not loaded. run cell 3 first.')
    ih_comparison_result = {}
else:
    print('invisible human: AI corpus vs climate change control corpus')
    print()

    ih_cols_in_df = [c for c in df.columns if c.startswith('ih_')]
    ai_ih = df[ih_cols_in_df]

    if 'title_lower' not in control_df.columns:
        control_df['title_lower'] = control_df['title'].str.lower().str.strip()
    control_ih = score_ih(control_df['title_lower'])

    print(f'AI corpus:      {len(ai_ih):,} headlines')
    print(f'control corpus: {len(control_ih):,} headlines')
    print()
    print(f'{"domain":<25} {"AI corpus":>11} {"climate":>11} {"diff":>8} {"AI lower?":>10}  sig')
    print('-' * 75)

    comparison_records = []

    for col in IH_COLS:
        label     = col.replace('ih_', '').replace('_', ' ').title()
        ai_rate   = ai_ih[col].mean() * 100 if col in ai_ih.columns else 0.0
        ctrl_rate = control_ih[col].mean() * 100
        diff      = ai_rate - ctrl_rate
        ai_lower  = 'YES' if diff < 0 else 'no'

        n_ai   = len(ai_ih)
        n_ctrl = len(control_ih)
        a = int(ai_ih[col].sum()) if col in ai_ih.columns else 0
        c = int(control_ih[col].sum())
        try:
            chi2, p_val, _, _ = chi2_contingency([[a, n_ai - a], [c, n_ctrl - c]])
            sig = 'p<0.001' if p_val < 0.001 else (f'p={p_val:.3f}' if p_val < 0.05 else 'n.s.')
        except Exception:
            sig = 'n/a'

        print(f'{label:<25} {ai_rate:>10.2f}% {ctrl_rate:>10.2f}% {diff:>+7.2f}% {ai_lower:>9}  {sig}')
        comparison_records.append({
            'domain':   label,
            'ai_pct':   round(float(ai_rate), 4),
            'ctrl_pct': round(float(ctrl_rate), 4),
            'diff_pct': round(float(diff), 4),
            'ai_lower': bool(diff < 0),
            'p_sig':    sig,
        })

    ai_any   = (ai_ih.sum(axis=1) > 0).mean() * 100
    ctrl_any = (control_ih.sum(axis=1) > 0).mean() * 100
    ai_lower_domains = sum(1 for r in comparison_records if r['ai_lower'])

    print()
    print(f'{"ANY domain (>= 1)":<25} {ai_any:>10.2f}% {ctrl_any:>10.2f}% {ai_any - ctrl_any:>+7.2f}%')

    comparison_df = pd.DataFrame(comparison_records)
    comparison_df.to_csv('outputs/ih_comparison.csv', index=False, encoding='utf-8')

    print()
    print('interpretation:')
    print(f'domains where AI scores lower than climate: {ai_lower_domains} / {len(IH_COLS)}')
    print()

    if ai_lower_domains >= 5:
        ih_finding_strength = 'strong'
        print('FINDING SUPPORTED: AI headlines score lower than climate on')
        print('the majority of IH domains. The 92% absence is NOT an artifact')
        print('of headline length -- it is specific to AI coverage.')
    elif ai_lower_domains >= 3:
        ih_finding_strength = 'moderate'
        print('FINDING PARTIALLY SUPPORTED: AI scores lower on some domains.')
        print(f'AI coverage underrepresents human-experience domains relative')
        print(f'to comparable news coverage in {ai_lower_domains} of 7 categories.')
    else:
        ih_finding_strength = 'weak'
        print('COMPRESSION EXPLAINS THE FINDING: AI and climate show similar')
        print('IH absence rates. Reframe as: headline compression reduces')
        print('human-experience vocabulary across news topics; AI coverage')
        print('is consistent with this general pattern.')

    print()
    print(f'to test whether the 92% absence rate is an artifact of headline')
    print(f'compression, we collected {len(control_df):,} climate change headlines')
    print(f'and applied the identical Invisible Human vocabulary. AI headlines')
    print(f'scored lower than climate headlines on {ai_lower_domains} of 7 domains')
    print(f'(any-domain rate: AI {ai_any:.1f}% vs climate {ctrl_any:.1f}%).')
    if ai_lower_domains >= 5:
        print('this supports the claim that the absence is specific to AI coverage,')
        print('not a general property of headline brevity.')
    else:
        print('headline compression accounts for some of the observed absence;')
        print('claims are framed accordingly.')

    # visualization
    fig, ax = plt.subplots(figsize=(11, 6))
    domains   = [r['domain'] for r in comparison_records]
    ai_vals   = [r['ai_pct'] for r in comparison_records]
    ctrl_vals = [r['ctrl_pct'] for r in comparison_records]
    x = np.arange(len(domains))
    w = 0.38
    ax.bar(x - w/2, ai_vals,   w, label='AI headlines (179K)',       color='#c0392b', alpha=0.85)
    ax.bar(x + w/2, ctrl_vals, w, label='climate change (control)', color='#2471a3', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(domains, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('% of headlines containing domain vocabulary')
    ax.set_title(
        'Invisible Human Framework: AI Coverage vs Climate Change Control Corpus\n'
        '(same vocabulary and pipeline — different topic)',
        fontsize=12, fontweight='bold'
    )
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('outputs/figures/12_ih_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print()
    print('saved: outputs/figures/12_ih_comparison.png')

    ih_comparison_result = {
        'n_ai':             int(len(ai_ih)),
        'n_control':        int(len(control_ih)),
        'control_topic':    'climate change',
        'ai_any_pct':       round(float(ai_any), 2),
        'ctrl_any_pct':     round(float(ctrl_any), 2),
        'ai_lower_domains': int(ai_lower_domains),
        'finding_strength': ih_finding_strength,
        'domains':          comparison_records,
    }

invisible human: AI corpus vs climate change control corpus

AI corpus:      179,372 headlines
control corpus: 4,106 headlines

domain                      AI corpus     climate     diff  AI lower?  sig
---------------------------------------------------------------------------
Grief Loss                      0.54%       0.90%   -0.36%       YES  p=0.003
Spirituality Meaning            0.61%       0.24%   +0.36%        no  p=0.004
Love Relationships              0.93%       0.78%   +0.15%        no  n.s.
Bodily Experience               0.61%       0.54%   +0.07%        no  n.s.
Dignity Purpose                 0.59%       0.07%   +0.51%        no  p<0.001
Community Belonging             0.31%       0.49%   -0.17%       YES  n.s.
Childhood Play                  4.16%       1.92%   +2.23%        no  p<0.001

ANY domain (>= 1)               7.55%       4.94%   +2.61%

interpretation:
domains where AI scores lower than climate: 2 / 7

COMPRESSION EXPLAINS THE FINDING: AI and climate show si

In [5]:
# export validation summary to outputs/validation_summary.json

validation_summary = {
    'generated':             datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'psi_hand_validation':   psi_validation_result,
    'ih_control_comparison': ih_comparison_result if 'ih_comparison_result' in dir() else {},
}

with open('outputs/validation_summary.json', 'w', encoding='utf-8') as fh:
    json.dump(validation_summary, fh, indent=2, ensure_ascii=False)

print('notebook 04 complete — validation summary')
print()

print('PSI hand-validation:')
if psi_validation_result.get('status') == 'complete':
    k      = psi_validation_result['kappa']
    interp = psi_validation_result['kappa_interp']
    n      = psi_validation_result['n_labeled']
    print(f'   kappa = {k:.3f}  ({interp}),  n = {n}')
    print(f"   report: 'Cohen kappa = {k:.2f} on {n} hand-labeled headlines'")
else:
    print(f"   status: {psi_validation_result.get('status', 'not run')}")
    print('   complete hand labeling and re-run cell 2.')
print()

print('invisible human control comparison:')
if 'ih_comparison_result' in dir() and ih_comparison_result:
    print(f"   control corpus: {ih_comparison_result['n_control']:,} climate change headlines")
    print(f"   AI any-domain rate:      {ih_comparison_result['ai_any_pct']:.1f}%")
    print(f"   climate any-domain rate: {ih_comparison_result['ctrl_any_pct']:.1f}%")
    print(f"   AI lower on {ih_comparison_result['ai_lower_domains']}/7 domains  "
          f"(finding strength: {ih_comparison_result['finding_strength']})")
else:
    print('   not run -- complete cells 3 and 4 first.')
print()

print('outputs written:')
for fname in [
    'psi_validation_template.csv',
    'psi_validation_results.csv',
    'control_corpus_headlines.csv',
    'ih_comparison.csv',
    'validation_summary.json',
]:
    path = f'outputs/{fname}'
    if os.path.exists(path):
        kb = os.path.getsize(path) // 1024
        print(f'   outputs/{fname:<38} {kb:>5,} KB')

fig_path = 'outputs/figures/12_ih_comparison.png'
if os.path.exists(fig_path):
    kb = os.path.getsize(fig_path) // 1024
    print(f'   outputs/figures/12_ih_comparison.png       {kb:>5,} KB')

print()

notebook 04 complete — validation summary

PSI hand-validation:
   kappa = 0.368  (fair),  n = 200
   report: 'Cohen kappa = 0.37 on 200 hand-labeled headlines'

invisible human control comparison:
   control corpus: 4,106 climate change headlines
   AI any-domain rate:      7.5%
   climate any-domain rate: 4.9%
   AI lower on 2/7 domains  (finding strength: weak)

outputs written:
   outputs/psi_validation_results.csv                21 KB
   outputs/control_corpus_headlines.csv           1,020 KB
   outputs/ih_comparison.csv                          0 KB
   outputs/validation_summary.json                    1 KB
   outputs/figures/12_ih_comparison.png          86 KB

